In [1]:
import pandas as pd

df = pd.read_csv("data/raw/t5_eligibility_by_language_201001_202609 (1).csv")

df.head()

/var/folders/kt/6s40djb10pbbvx2x9lnztb9r0000gn/T/ipykernel_51565/956001093.py:3: DtypeWarning: Columns (0: Date of Data) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/raw/t5_eligibility_by_language_201001_202609 (1).csv")


,Month of Eligibility,County,Language,Total Eligibles,Annotation Code,Annotation Description,Status,Date of Data,Difference
0,2026-08,Alameda,ASL,132,NaN,NaN,P,11AUG2026,NaN
1,2026-08,Alameda,Arabic,"2,724",NaN,NaN,P,11AUG2026,NaN
2,2026-08,Alameda,Armenian,35,NaN,NaN,P,11AUG2026,NaN
3,2026-08,Alameda,Cambodian,715,NaN,NaN,P,11AUG2026,NaN
4,2026-08,Alameda,Cantonese,"23,765",NaN,NaN,P,11AUG2026,NaN


In [2]:
df.shape

(251535, 9)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 251535 entries, 0 to 251534
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Month of Eligibility    251535 non-null  str    
 1   County                  251535 non-null  str    
 2   Language                251535 non-null  str    
 3   Total Eligibles         157214 non-null  str    
 4   Annotation Code         94324 non-null   float64
 5   Annotation Description  94524 non-null   str    
 6   Status                  251535 non-null  str    
 7   Date of Data            113396 non-null  str    
 8   Difference              0 non-null       float64
dtypes: float64(2), str(7)
memory usage: 27.4 MB


In [4]:
df.columns.tolist()

['Month of Eligibility',
 'County',
 'Language',
 'Total Eligibles',
 'Annotation Code',
 'Annotation Description',
 'Status',
 'Date of Data',
 'Difference']

In [ ]:
import os
import pandas as pd

raw_path = "data/raw/t5_eligibility_by_language_201001_202609 (1).csv"
clean_path = "data/cleaned/medi_cal_eligibility_language_clean.csv"

# Load the raw data
raw = pd.read_csv(raw_path)

# Normalize column names and clean obvious formatting issues
raw.columns = [col.strip() for col in raw.columns]
raw = raw.rename(columns={
    "Month of Eligibility": "month",
    "County": "county",
    "Language": "language",
    "Total Eligibles": "total_eligibles",
    "Annotation Code": "annotation_code",
    "Annotation Description": "annotation_description",
    "Status": "status",
    "Date of Data": "date_of_data",
    "Difference": "difference",
})

# Convert numeric text like "2,724" to integers and keep missing values as NaN
raw["total_eligibles"] = raw["total_eligibles"].astype(str)
raw["total_eligibles"] = raw["total_eligibles"].str.replace(",", "", regex=False)
raw["total_eligibles"] = pd.to_numeric(raw["total_eligibles"], errors="coerce")

# Parse month and date fields to standard datetime values
raw["month"] = pd.to_datetime(raw["month"], format="%Y-%m", errors="coerce")
raw["date_of_data"] = pd.to_datetime(raw["date_of_data"], format="%d%b%Y", errors="coerce")

# Keep only rows that are useful for analysis: county-language counts with valid values
clean = raw[
    raw["county"].notna() &
    raw["language"].notna() &
    raw["total_eligibles"].notna() &
    raw["status"].notna()
].copy()

# Remove row-level annotation summaries that are not county-language records
clean = clean[~clean["annotation_description"].fillna("").str.contains("Statewide Total by MOE", na=False)]

# Retain only final counts, excluding preliminary records if the goal is a clean analytic table
clean = clean[clean["status"] == "F"].copy()

# Optional: standardize text values
clean["county"] = clean["county"].str.strip()
clean["language"] = clean["language"].str.strip()
clean["status"] = clean["status"].str.strip()

# Sort for predictability and save cleaned data
clean = clean.sort_values(["month", "county", "language"]).reset_index(drop=True)
os.makedirs(os.path.dirname(clean_path), exist_ok=True)
clean.to_csv(clean_path, index=False)

print(f"Raw rows: {len(raw)}")
print(f"Cleaned rows: {len(clean)}")
print(f"Saved cleaned file: {clean_path}")
print(clean.head().to_string(index=False))